# Stage 3 -- Dependency-aware deep detectors on S1 (E1, E2)
Trains LGC, LAT, LAT-mask, BR-trunk and the re-implemented WIDENNET on all views under
S1; hyper-parameter search (1 seed) then final runs (3 seeds); family-token ablation on
F_all; saves test probabilities for Stage 5. **Accelerator: GPU (T4/P100).**
**Attach:** `scvd-lib`, `dive-smart-contracts`, `scvd-stage1-eda`, `scvd-stage2-baselines`.

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd
LIB = '/kaggle/input/scvd-lib'
if not os.path.isdir(LIB):
    LIB = os.environ.get('SCVD_LIB', os.path.abspath('..'))   # local fallback
sys.path.insert(0, LIB)
import scvd
root, log = scvd.setup_stage('stage3_deep')
SEEDS = (0,) if scvd.quick() else scvd.RNG_SEEDS
log.info('seeds=%s quick=%s', SEEDS, scvd.quick())

In [ ]:
pre, post, lab, tools = scvd.load_dive(log)
fam = scvd.family_columns(pre, post)
for k, v in fam.items():
    log.info('family %-18s %3d features', k, len(v))
for v in scvd.VIEWS:
    log.info('view F_%-4s %3d features', v, len(scvd.view_columns(fam, v)))

In [ ]:
splits, sdf = scvd.load_splits() or (scvd.build_splits(pre, lab, 0), None)
GRID = [{'lr': lr, 'dm': dm, 'layers': L, 'epochs': (30 if scvd.quick() else 120),
         'batch': 256, 'tau': tau}
        for lr in (1e-3, 3e-4) for dm in (128, 256) for L in (2, 3) for tau in (0.1, 0.3)]
TRIALS = 4 if scvd.quick() else len(GRID)

In [ ]:
def build(model_name, vf, M, cfg, n_op):
    if model_name == 'LGC':      return scvd.LGC(vf, M, dm=cfg['dm'], tau=cfg['tau'])
    if model_name == 'LAT':      return scvd.LAT(vf, dm=cfg['dm'], layers=cfg['layers'])
    if model_name == 'LAT-mask': return scvd.LAT(vf, dm=cfg['dm'], layers=cfg['layers'], mask_ll=True)
    if model_name == 'BR-trunk': return scvd.LAT(vf, dm=cfg['dm'], layers=cfg['layers'], trunk_only=True)
    if model_name == 'WIDENNET': return scvd.WIDENNET(n_op, dm=cfg['dm'])

rows, preds = [], {}
for view in scvd.VIEWS:
    d = scvd.make_data(pre, post, lab, fam, view, *splits['S1'])
    M = scvd.cooccurrence(d['y_tr'])
    op_cols = [c for n in scvd.view_families(fam, view) if 'opcodes' in n for c in fam[n]]
    n_op = len(op_cols)
    for name in ['LGC', 'LAT', 'LAT-mask', 'BR-trunk', 'WIDENNET']:
        kind = 'op' if name == 'WIDENNET' else 'fam'
        if kind == 'op':  # flat opcode matrix views
            Xop_tr = np.hstack([d['x_tr'][n] for n in d['x_tr'] if 'opcodes' in n])
            Xop_va = np.hstack([d['x_va'][n] for n in d['x_va'] if 'opcodes' in n])
            Xop_te = np.hstack([d['x_te'][n] for n in d['x_te'] if 'opcodes' in n])
            duse = {'x_tr': Xop_tr, 'x_va': Xop_va, 'x_te': Xop_te,
                    'y_tr': d['y_tr'], 'y_va': d['y_va'], 'y_te': d['y_te']}
        else:
            duse = d
        best_cfg, best_vm = GRID[0], -1
        for cfg in GRID[:TRIALS]:
            mdl = build(name, d['fams'] if kind == 'fam' else None, M, cfg, n_op)
            _, hist = scvd.train_torch(mdl, duse, cfg, log, kind)
            vm = max(h['val_macro_f1'] for h in hist)
            if vm > best_vm: best_vm, best_cfg = vm, cfg
        for seed in SEEDS:
            np.random.seed(seed)
            mdl = build(name, d['fams'] if kind == 'fam' else None, M, best_cfg, n_op)
            pt, hist = scvd.train_torch(mdl, duse, best_cfg, log, kind)
            pv = scvd.predict(mdl, duse, 'va', kind)
            th = scvd.tune_thresholds(d['y_va'], pv)
            m = scvd.all_metrics(d['y_te'], (pt >= th).astype(int), pt)
            rows.append({'model': name, 'view': view, 'split': 'S1', 'seed': seed, **best_cfg, **m})
            scvd.append_results(root, 'stage3_deep', [rows[-1]])   # checkpoint per run
            preds[(name, view, seed)] = (d['ids_te'], pt, th)
            log.info('%-9s F_%-4s seed%d macro_f1=%.4f', name, view, seed, m['macro_f1'])
np.savez_compressed(f'{root}/results/stage3_test_predictions.npz',
                    **{f'{n}__{v}__{s}__ids': i for (n, v, s), (i, _p, _t) in preds.items()},
                    **{f'{n}__{v}__{s}__probs': p for (n, v, s), (_i, p, _t) in preds.items()})
json.dump({f'{n}|{v}|{s}': list(map(float, t)) for (n, v, s), (_i, _p, t) in preds.items()},
          open(f'{root}/results/stage3_thresholds.json', 'w'))

In [ ]:
drop = {}
d = scvd.make_data(pre, post, lab, fam, 'all', *splits['S1'])
M = scvd.cooccurrence(d['y_tr'])
base = {}
for name in ['LGC', 'LAT']:
    cfg = {'lr': 1e-3, 'dm': 128, 'layers': 3, 'epochs': 60 if scvd.quick() else 120,
           'batch': 256, 'tau': 0.2}
    mdl = build(name, d['fams'], M, cfg, 0)
    pt, _h = scvd.train_torch(mdl, d, cfg, log)
    base[name] = scvd.all_metrics(d['y_te'], (pt >= 0.5).astype(int), pt)['macro_f1']
    for f in d['fams']:
        vf2 = {k: v for k, v in d['fams'].items() if k != f}
        d2 = {k: v for k, v in d.items() if not k.startswith('x_')}
        d2['fams'] = vf2
        for part in ('tr', 'va', 'te'):
            d2['x_' + part] = {k: v for k, v in d['x_' + part].items() if k != f}
        mdl2 = build(name, vf2, M, cfg, 0)
        pt2, _h2 = scvd.train_torch(mdl2, d2, cfg, log)
        drop.setdefault(f, {})[name] = base[name] - scvd.all_metrics(
            d['y_te'], (pt2 >= 0.5).astype(int), pt2)['macro_f1']
json.dump(drop, open(f'{root}/results/stage3_family_ablation.json', 'w'), indent=1)
scvd.plot_family_ablation(drop, root, log)

In [ ]:
summ, sp = scvd.write_summary(root, 'stage3_deep')
scvd.plot_model_macro_f1(summ, root, 's3_deep_macro_f1', log)
scvd.plot_perlabel_f1(summ, ['LGC', 'LAT', 'LAT-mask', 'BR-trunk'], root, 's3_deep_perlabel', log)
scvd.tex_table(summ, f'{root}/tables/stage3_deep.tex',
               ['macro_f1', 'micro_f1', 'macro_ap', 'subset_acc'],
               'Deep detectors on S1 (mean over seeds).', 'tab:stage3deep')
summ.round(4)

In [ ]:
url = scvd.push_dataset(root, 'stage3_deep', 'scvd-stage3-deep', 'SCVD Stage 3: dependency-aware deep detectors on DIVE', log)
if url:
    print('DATASET:', url)
else:
    print('push skipped (no credentials); artifacts remain under', root)